## Transformer sentiment analysis

Here I'm taking the code from [this tutorial](https://www.datacamp.com/tutorial/building-a-transformer-with-py-torch) and adapting it to do sentiment analysis on movie reviews

In [34]:
#Import all the things
import math
import copy

import kagglehub
import pandas as pd
from sentence_transformers import SentenceTransformer
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
from torch.utils.data import DataLoader

In [35]:
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-of-50k-movie-reviews


In [36]:
# We could probably do without Pandas
df = pd.read_csv(path + "/IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [37]:
device = "cuda" if torch.cuda.is_available() else "cpu"
embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

Divide the data: 80% for training

In [38]:
# divide df into df_train and df_valid
df_train = df.sample(frac=0.8, random_state=0)
df_valid = df.drop(df_train.index)

Create sentence encodings for each review.

In [79]:
#Get encodings for df['review']
# Process in batches to avoid memory issues
batch_size = 1000 # Adjust batch size as needed
training_embeddings = []
training = df_train['review'].tolist()
for i in range(0, len(training), batch_size):
    batch = training[i:i+batch_size]
    training_embeddings.extend(embedding_model.encode(batch))
    print(f'Processed batch { i//batch_size+1 } of { len(training)//batch_size}\n' if (i % 10*batch_size == 0) else "", end="\r")

training_embeddings = torch.tensor(training_embeddings)
print(training_embeddings.shape)

Processed batch 1 of 40
Processed batch 2 of 40
Processed batch 3 of 40
Processed batch 4 of 40
Processed batch 5 of 40
Processed batch 6 of 40
Processed batch 7 of 40
Processed batch 8 of 40
Processed batch 9 of 40
Processed batch 10 of 40
Processed batch 11 of 40
Processed batch 12 of 40
Processed batch 13 of 40
Processed batch 14 of 40
Processed batch 15 of 40
Processed batch 16 of 40
Processed batch 17 of 40
Processed batch 18 of 40
Processed batch 19 of 40
Processed batch 20 of 40
Processed batch 21 of 40
Processed batch 22 of 40
Processed batch 23 of 40
Processed batch 24 of 40
Processed batch 25 of 40
Processed batch 26 of 40
Processed batch 27 of 40
Processed batch 28 of 40
Processed batch 29 of 40
Processed batch 30 of 40
Processed batch 31 of 40
Processed batch 32 of 40
Processed batch 33 of 40
Processed batch 34 of 40
Processed batch 35 of 40
Processed batch 36 of 40
Processed batch 37 of 40
Processed batch 38 of 40
Processed batch 39 of 40
Processed batch 40 of 40
torch.Siz

In [40]:
validation_embeddings = []
validation = df_valid['review'].tolist()
for i in range(0, len(validation), batch_size):
    batch = validation[i:i+batch_size]
    validation_embeddings.extend(embedding_model.encode(batch))
    print(f'Processed batch { i//batch_size+1 } of { len(validation)//batch_size}\n' if (i % 10*batch_size == 0) else "", end="\r")

validation_embeddings = torch.tensor(validation_embeddings)
print(validation_embeddings.shape)

Processed batch 1 of 10
Processed batch 2 of 10
Processed batch 3 of 10
Processed batch 4 of 10
Processed batch 5 of 10
Processed batch 6 of 10
Processed batch 7 of 10
Processed batch 8 of 10
Processed batch 9 of 10
Processed batch 10 of 10
torch.Size([10000, 384])


In [80]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadAttention, self).__init__()
        # Ensure that the model dimension (d_model) is divisible by the number of heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        # Initialize dimensions
        self.d_model = d_model # Model's dimension
        self.num_heads = num_heads # Number of attention heads
        self.d_k = d_model // num_heads # Dimension of each head's key, query, and value

        # Linear layers for transforming inputs
        self.W_q = nn.Linear(d_model, d_model) # Query transformation
        self.W_k = nn.Linear(d_model, d_model) # Key transformation
        self.W_v = nn.Linear(d_model, d_model) # Value transformation
        self.W_o = nn.Linear(d_model, d_model) # Output transformation

    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        # Calculate attention scores
        attn_scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)

        # Apply mask if provided (useful for preventing attention to certain parts like padding)
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)

        # Softmax is applied to obtain attention probabilities
        attn_probs = torch.softmax(attn_scores, dim=-1)

        # Multiply by values to obtain the final output
        output = torch.matmul(attn_probs, V)
        return output

    def split_heads(self, x):
        # Reshape the input to have num_heads for multi-head attention
        batch_size, seq_length, d_model = x.size()
        return x.view(batch_size, seq_length, self.num_heads, self.d_k).transpose(1, 2)

    def combine_heads(self, x):
        # Combine the multiple heads back to original shape
        batch_size, _, seq_length, d_k = x.size()
        return x.transpose(1, 2).contiguous().view(batch_size, seq_length, self.d_model)

    def forward(self, Q, K, V, mask=None):
        # Apply linear transformations and split heads
        Q = self.split_heads(self.W_q(Q))
        K = self.split_heads(self.W_k(K))
        V = self.split_heads(self.W_v(V))

        # Ensure mask is on the same device as Q, K, V if it exists
        if mask is not None:
            mask = mask.to(Q.device)

        # Perform scaled dot-product attention
        attn_output = self.scaled_dot_product_attention(Q, K, V, mask)

        # Combine heads and apply output transformation
        output = self.W_o(self.combine_heads(attn_output))
        return output

In [43]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(PositionWiseFeedForward, self).__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.relu = nn.ReLU()

    def forward(self, x):
        # Ensure input x is on the correct device before applying linear layers
        return self.fc2(self.relu(self.fc1(x.to(self.fc1.weight.device))))

In [44]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_seq_length, d_model)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

In [45]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout):
        super(EncoderLayer, self).__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask):
        # Ensure input x is on the correct device before passing to sub-modules
        x = x.to(self.norm1.weight.device)

        attn_output = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

In [62]:
class Transformer(nn.Module):
    def __init__(self, d_model, num_heads, num_layers, d_ff, dropout):
        super(Transformer, self).__init__()
        # Removed embedding layers and positional encoding
        # self.encoder_embedding = nn.Embedding(src_vocab_size, d_model)
        # self.decoder_embedding = nn.Embedding(tgt_vocab_size, d_model)
        # self.positional_encoding = PositionalEncoding(d_model, max_seq_length)

        self.encoder_layers = nn.ModuleList([EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)])
        # self.decoder_layers = nn.ModuleList([DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]) # Decoder is not needed for classification

        # Adjusted the output layer to reflect the change in target (sentiment label)
        self.fc = nn.Linear(d_model, 2) # Output size is 2 for binary sentiment classification
        self.dropout = nn.Dropout(dropout)

    def generate_mask(self, src, tgt):
        # Masks are not needed when using pre-computed embeddings and a single target
        src_mask = None
        tgt_mask = None
        return src_mask, tgt_mask

    def forward(self, src, tgt):
        # Directly use the input embeddings
        # Assuming input `src` is [batch_size, 1, d_model]
        src_mask = None # No mask needed for sequence length 1

        enc_output = src
        for enc_layer in self.encoder_layers:
            enc_output = enc_layer(enc_output, src_mask)

        # The output of the encoder is [batch_size, 1, d_model]
        # We need to flatten this to [batch_size, d_model] for the linear classifier
        output = self.fc(enc_output.squeeze(1)) # Squeeze to remove the sequence length dimension

        return output

In [72]:
d_model = 384
num_heads = 32
num_layers = 6
d_ff = 64
dropout = 0.1

transformer = Transformer(d_model, num_heads, num_layers, d_ff, dropout)

Not sure why we're overfitting.

In [75]:
# Create DataLoader for validation set
x_valid = validation_embeddings
y_valid = torch.tensor(df_valid['sentiment'].map({'positive': 1, 'negative': 0}).values)
val_dataset = data.TensorDataset(x_valid, y_valid)
val_dataloader = DataLoader(val_dataset, batch_size=100, shuffle=True) # You can adjust the batch_size

In [74]:
# Create Dataset and DataLoader
x_train = training_embeddings
y_train = torch.tensor(df_train['sentiment'].map({'positive': 1, 'negative': 0}).values, dtype=torch.long) # Ensure target is Long type
train_dataset = data.TensorDataset(x_train, y_train)
train_dataloader = DataLoader(train_dataset, batch_size=2500, shuffle=True) # Increased batch size

criterion = nn.CrossEntropyLoss() # CrossEntropyLoss expects target of type Long
optimizer = optim.Adam(transformer.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9) # Reduced learning rate

# Check if CUDA is available and move model and criterion to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
transformer.to(device)
criterion.to(device)


print("Starting training...")
transformer.train()

for epoch in range(25):
    total_train_loss = 0
    for batch_src, batch_tgt in train_dataloader:
        # Move batch data to GPU and reshape source to [batch_size, 1, d_model]
        batch_src = batch_src.to(device).unsqueeze(1) # Add sequence length dimension
        # Target is already [batch_size] and is used as is by CrossEntropyLoss
        batch_tgt = batch_tgt.to(device)


        optimizer.zero_grad()
        output = transformer(batch_src, batch_tgt) # Pass batch_tgt, though it's not used in the redefined forward
        loss = criterion(output, batch_tgt) # CrossEntropyLoss expects output [batch_size, num_classes] and target [batch_size]
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_dataloader)

    # Evaluate on validation set
    transformer.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch_src, batch_tgt in val_dataloader:
            batch_src = batch_src.to(device).unsqueeze(1)
            batch_tgt = batch_tgt.to(device)
            val_output = transformer(batch_src, batch_tgt)
            val_loss = criterion(val_output, batch_tgt)
            total_val_loss += val_loss.item()
    avg_val_loss = total_val_loss / len(val_dataloader)

    print(f"Epoch: {epoch+1}, Training Loss: {avg_train_loss:.4f}, Validation Loss: {avg_val_loss:.4f}")
    transformer.train() # Set back to training mode

Using device: cuda
Starting training...
Epoch: 1, Training Loss: 0.4327, Validation Loss: 0.3998
Epoch: 2, Training Loss: 0.3899, Validation Loss: 0.3935
Epoch: 3, Training Loss: 0.3794, Validation Loss: 0.3854
Epoch: 4, Training Loss: 0.3730, Validation Loss: 0.3847
Epoch: 5, Training Loss: 0.3685, Validation Loss: 0.3836
Epoch: 6, Training Loss: 0.3645, Validation Loss: 0.3826
Epoch: 7, Training Loss: 0.3579, Validation Loss: 0.3871
Epoch: 8, Training Loss: 0.3501, Validation Loss: 0.3847
Epoch: 9, Training Loss: 0.3435, Validation Loss: 0.3847
Epoch: 10, Training Loss: 0.3359, Validation Loss: 0.3839
Epoch: 11, Training Loss: 0.3286, Validation Loss: 0.3908
Epoch: 12, Training Loss: 0.3190, Validation Loss: 0.3863
Epoch: 13, Training Loss: 0.3131, Validation Loss: 0.3920
Epoch: 14, Training Loss: 0.3022, Validation Loss: 0.3952
Epoch: 15, Training Loss: 0.2924, Validation Loss: 0.3947
Epoch: 16, Training Loss: 0.2828, Validation Loss: 0.4011
Epoch: 17, Training Loss: 0.2742, Validat

In [76]:
transformer.eval()

total_loss = 0
with torch.no_grad():
    for batch_src, batch_tgt in val_dataloader:
        # Move batch data to GPU and reshape source to [batch_size, 1, d_model]
        batch_src = batch_src.to(device).unsqueeze(1) # Add sequence length dimension
        # Target is already [batch_size]
        batch_tgt = batch_tgt.to(device)

        val_output = transformer(batch_src, batch_tgt) # Pass batch_tgt, though it's not used in the redefined forward
        loss = criterion(val_output, batch_tgt) # CrossEntropyLoss expects output [batch_size, num_classes] and target [batch_size]
        total_loss += loss.item()

avg_val_loss = total_loss / len(val_dataloader)
print(f"Validation Loss: {avg_val_loss}")

Validation Loss: 0.46651050075888634


In [77]:
# Calculate accuracy on validation set
transformer.eval() # Set the model to evaluation mode
correct_predictions = 0
total_predictions = 0

with torch.no_grad():
    for batch_src, batch_tgt in val_dataloader:
        batch_src = batch_src.to(device).unsqueeze(1)
        batch_tgt = batch_tgt.to(device)

        val_output = transformer(batch_src, batch_tgt)
        _, predicted = torch.max(val_output.data, 1)
        total_predictions += batch_tgt.size(0)
        correct_predictions += (predicted == batch_tgt).sum().item()

accuracy = correct_predictions / total_predictions
print(f"Validation Accuracy: {accuracy:.4f}")

Validation Accuracy: 0.8251


In [78]:
def run_model(inputdim, outputdim, hiddendim1, hiddenact1, hiddendim2, hiddenact2, hiddendim3=None, hiddenact3=None):
# Create Three Layered Neural Network
      print(f"First hidden layer: {hiddendim1} nodes, {hiddenact1.__name__} activation.")
      print(f"Second hidden layer: {hiddendim2} nodes, {hiddenact2.__name__} activation.")

      if hiddendim3 is not None:

            print(f"Third hidden layer: {hiddendim3} nodes, {hiddenact3.__name__} activation.")
            model = ThreeLayeredNN(inputdim, hiddendim1, hiddenact1,
                              hiddendim2, hiddenact2,
                              hiddendim3, hiddenact3, outputdim)
      else:
            model = TwoLayeredNN(inputdim, hiddendim1, hiddenact1,
                                    hiddendim2, hiddenact2, outputdim)

      # Cross Entropy Loss
      error = nn.CrossEntropyLoss()

      # SGD Optimizer
      learning_rate = 0.05
      momentum_rate = 0.9
      #optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
      optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=momentum_rate)

      # Let's train the model.
      print('Number of Epochs:',num_epochs)
      count = 0
      loss_list = []
      iteration_list = []
      accuracy_list = []
      for epoch in range(num_epochs):
        #print('Epoch Number',epoch)
        for i, (images, labels) in enumerate(train_loader):

            train = Variable(images.view(-1, 28*28))
            labels = Variable(labels)

            # Clear gradients
            optimizer.zero_grad()
            # Forward propagation
            outputs = model(train)
            # Calculate the Cross entropy loss (built in softmax)
            loss = error(outputs, labels)
            # Back propagation
            loss.backward()
            # Update parameters
            optimizer.step()

            #Print out accuracy and loss. (And store for later graphs)
            count += 1
            if count % 50 == 0:
                # Calculate Accuracy
                correct = 0
                total = 0
                # Predict test dataset
                for images, labels in test_loader:
                    test = Variable(images.view(-1, 28*28))

                    # Forward propagation
                    outputs = model(test)
                    # Get predictions from the maximum value
                    predicted = torch.max(outputs.data, 1)[1]
                    # Total number of labels
                    total += len(labels)
                    # Total correct predictions
                    correct += (predicted == labels).sum()

                accuracy = 100 * correct / float(total)

                # store loss and iteration
                loss_list.append(loss.data)
                iteration_list.append(count)
                accuracy_list.append(accuracy)

      print('Iteration: {}  Loss: {}  Accuracy: {} %'.format(count, loss.data, accuracy))
